<div class="alert alert-block alert-success">


# 02 — cleaning-transforming pipeline

## Goal of this notebook
### Implement the data cleaning\transforming for better data quality:
#### issues found in previous notebook:
- add a transformed numeric_net_transfer_record column in clubs table to show the pure number without any + ,- ,k or m symbol 
- splitting player_code column in players table to first_name and last_name, which is not the best approach we can do it per-query if needed
- replacing the height_in_cm column in players table lower then 100cm with the median or mean 
- handling the '-1' minute but the best approach is to live it at -1 because 90 percent of the '-1' minute columns are shootout events and replacing them with median or mean will cause biased analysis on shootout events

#### issues to actually implement:
- add a transformed numeric_net_transfer_record column in clubs table to show the pure number without any + ,- ,k or m symbol 
- replacing the height_in_cm column in players table lower then 100cm with the median or mean 

##### the changes are going to take place directly on the database not the raw data

In [1]:
# importing the necessaries
import sys
import os
# Adding the root to the path to use utils folder
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.db_utils import run_query 
from utils.custom_plots import distribution_plot, outlier_plot


import plotly.io as pio
pio.renderers.default = "png"  # drop these 2 lines if you want interactive charts locally

<div class="alert alert-block alert-success">

# Implementing the first issue:
- add a transformed numeric_net_transfer_record column in clubs table to show the pure number without any + ,- ,k or m symbol 


In [ ]:
first_q = '''
ALTER TABLE clubs 
ADD COLUMN net_transfer_total NUMERIC GENERATED ALWAYS AS (
  CASE
    WHEN trim(net_transfer_record) = '+-0' THEN 0
    WHEN trim(net_transfer_record) ILIKE '%m' THEN
      CAST(substring(trim(net_transfer_record), '-?\d+\.?\d*') AS NUMERIC) * 1000000
    WHEN trim(net_transfer_record) ILIKE '%k' THEN
      CAST(substring(trim(net_transfer_record), '-?\d+\.?\d*') AS NUMERIC) * 1000
    ELSE CAST(substring(trim(net_transfer_record), '-?\d+\.?\d*') AS NUMERIC)
  END
) STORED;
'''
run_query(first_q)

In [ ]:
# checking the result:
